In [ ]:
import h5py
import numpy as np
import scanpy as sc
import scipy.sparse as sp_sparse
import os
import pandas as pd
import gzip
import time
import re
#from pyensembl import EnsemblRelease
import pwd
import itertools
#### scanpy parameters
import omicverse as ov
import sys
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor="white")
import harmonypy as hm
import bbknn
import datetime

In [ ]:
adata = sc.read_h5ad('external_inputs/adata_meta.h5ad')
gse_counts = adata.obs['tissue'].value_counts()
print("原始GSE统计:")
print(gse_counts.head())

# 筛选细胞数大于50的GSE
valid_gses = gse_counts[gse_counts > 50].index.tolist()
print(f"\n细胞数大于50的GSE数量: {len(valid_gses)}")
print(f"细胞数小于等于50的GSE数量: {len(gse_counts) - len(valid_gses)}")

# 创建子集
adata_sub = adata[adata.obs['tissue'].isin(valid_gses)].copy()


In [ ]:
adata = adata_sub
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.X = np.nan_to_num(adata.X)
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
sc.pp.pca(adata, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(adata, n_pcs=50)
sc.tl.umap(adata)
# 需要重新矫正批次
print(datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


In [ ]:
print("Start BBKNN")
bbknn.bbknn(adata, batch_key='tissue', n_pcs=50)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.8,key_added='res0.8')
sc.tl.leiden(adata, resolution=1 ,key_added='res1')
sc.tl.leiden(adata, resolution=1.6 ,key_added='res1.6')
sc.tl.leiden(adata, resolution=2 ,key_added='res2')


In [ ]:
adata.write('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_metacell_tissuebatch_new.h5ad')

In [ ]:
sc.pl.umap(adata, color=['leiden','res2','Celltype','Celltype_general','tissue'],  show=True, ncols =2, wspace=0.5,hspace=0.2 ) 



In [ ]:
# adata = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_10x_TissueNoSplit_degs.h5ad')
species ='Human'
scsa=ov.single.pySCSA(adata=adata,foldchange=1.5,celltype='normal',target='cellmarker',
                  # tissue='Kidney',
                  species=species)  ## model_path='temp/pySCSA_2023_v2_plus.db', 如果没有model_path就会自动下载
#scsa.get_model_tissue(species=species)
res=scsa.cell_anno(clustertype='leiden', ##注释依据的簇名，存放在adata.obs
               cluster='all',##需要注释的cluster，可以设置为list: ['1','2']表示只注释簇1和簇2
               rank_rep=True)##是否需要重新计算差异表达基因   ##如果已经运算了sc.tl.rank_genes_groups，可以设置成False停止重复计算
res = res.dropna(subset='Z-score')
scsa.result = res  ### df.dropna()去除df中有nan的所有行; df.dropna(subset=['colB']),去除df中B列为nan的所有行
# scsa.cell_anno_print() ## 这个函数在这print会出问题
max_zscore_indices = scsa.result.groupby('Cluster')['Z-score'].idxmax()
max_zscore_rows = scsa.result.loc[max_zscore_indices]
cluster_cell_types = max_zscore_rows.set_index('Cluster')['Cell Type'].to_dict()
max_zscore_rows.rename(columns={'Cluster':'leiden'}, inplace=True)
meta_data = adata.obs
meta_data['leiden'] = meta_data['leiden'].astype(int)
merged_df = pd.merge(meta_data, max_zscore_rows, on='leiden', how='left')
merged_df.index = meta_data.index
adata.obs = merged_df



In [ ]:
adata = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation.h5ad') 



In [ ]:
sc.pl.umap(adata, color=['leiden','Celltype','Celltype_general','tissue'],  show=True, ncols =2, wspace=0.5,hspace=0.2 ) 


In [ ]:
adata_pan = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_panCT.h5ad') 


In [ ]:
sc.pl.umap(adata_pan, color=['leiden','Celltype','Celltype_general','tissue'],  show=True, ncols =2, wspace=0.5,hspace=0.2 ) 


In [ ]:
adata.obs[['leiden','Celltype','Celltype_general','Celltype_sub']].drop_duplicates().sort_values('Celltype_general')


In [ ]:
adata_pan.obs[['leiden','Celltype','Celltype_general','Celltype_sub','tissue_type']].drop_duplicates().sort_values('Celltype_general')


In [ ]:
meta_data = adata.obs
meta_data['tissue'] = meta_data['tissue'].astype(str)
meta_data['leiden'] = meta_data['leiden'].astype(str)
meta_data['Celltype'] = meta_data['Celltype'].astype(str)
meta_data['Celltype_general'] = meta_data['Celltype_general'].astype(str)
meta_data['Celltype_sub'] = meta_data['Celltype_sub'].astype(str)
meta_data.to_csv('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_meta.tsv',sep='\t')


In [ ]:
adata.obs['tissue_type'] = adata.obs['Celltype'].apply(lambda x: x.split('_')[2])
adata.obs['tissue_type'].value_counts()


In [ ]:
adata_pan = adata[adata.obs['tissue_type']=='pan']
adata_pan


In [ ]:
os.getcwd()
os.chdir('external_inputs/wangyuting')

In [ ]:
adata_pan.write('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_panCT.h5ad')


In [ ]:
adata_pan.raw


In [ ]:
########## 接下来画那 8个 signature score 的柱状图
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
# 设置Nature/Cell期刊风格的参数
plt.style.use('default')
rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})
# 1. 提取数据框并添加Celltype信息
adata_meta = adata.obs[['leiden','Celltype_general','Celltype','MCs', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT']].copy()
# rename_dict = {
#     'score_Tip': 'Tip',
#     'score_antigen_pre': 'AP',
#     'score_stalk': 'Stalk'
# }
# adata_meta = adata_meta.rename(columns=rename_dict) 
adata_meta['Celltype'] = adata_meta['Celltype'].astype(str) + '_C' + adata_meta['leiden'].astype(str)
# 定义leiden到Celltype的映射字典

# 确保leiden列是字符串类型，方便处理
adata_meta['leiden'] = adata_meta['leiden'].astype(str)
# 3. 按照leiden分组计算平均值
# 注意：这里修正了列名大小写问题
feature_columns = ['MCs', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT']      ##  ['Tip', 'AP', 'Stalk'], 可以替换
# 检查列名是否存在，确保大小写匹配
available_columns = [col for col in feature_columns if col in adata_meta.columns]
grouped_mean = adata_meta.groupby(['Celltype'])[available_columns].mean().reset_index()
grouped_mean['leiden'] = grouped_mean['Celltype'].str.split('_C').str[-1].astype(int)
grouped_mean = grouped_mean.sort_values(['Celltype','leiden'])
print(grouped_mean[['leiden', 'Celltype']].to_string(index=False))
# 5. 可视化设置
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}")
# 设置子图布局：4行10列
n_rows = 6
n_cols = 8
n_subplots = n_rows * n_cols
# 如果leiden数量少于子图数量，调整布局
if n_leiden < n_subplots:
    n_cols = int(np.ceil(n_leiden / n_rows))
    n_subplots = n_rows * n_cols

colors =  plt.cm.Set2(np.linspace(0, 1, len(available_columns)))
# 6. 创建图形
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2*n_cols, 1.2*n_rows))
axes = axes.flatten()
# 设置图形标题
fig.suptitle('Signature score of subEndo markers',  # Signature score of subFibro(long list)
             fontsize=12, fontweight='bold', y=1.02)

# 7. 绘制每个leiden的柱状图（添加数值标签）
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):
        break
    
    leiden = row.leiden
    celltype = row.Celltype
    ax = axes[i]
    
    # 提取当前leiden的数据
    leiden_data = grouped_mean.loc[grouped_mean['leiden'] == leiden, available_columns].iloc[0]
    
    # 创建柱状图
    x_pos = np.arange(len(available_columns))
    bars = ax.bar(x_pos, leiden_data.values, color=colors, edgecolor='black', linewidth=0.5)
    
    # 添加数值标签到每个柱子顶部
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        # 确定标签位置
        height = bar.get_height()
        
        # 如果值太小，将标签放在柱子内部顶部
        if height < grouped_mean[available_columns].values.max() * 0.05:
            # 小值：标签放在柱子内部，白色字体
            text_y = height * 0.8  # 放在柱子高度的80%位置
            color = 'black' # 小的值也还是 black
            va = 'bottom'
        else:
            # 正常值：标签放在柱子顶部上方
            text_y = height + grouped_mean[available_columns].values.max() * 0.01  # 稍微高于柱子
            color = 'black'
            va = 'bottom'
        
        # 格式化数值显示（根据数值大小选择格式）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加文本标签
        ax.text(
            bar_idx, text_y, text,
            ha='center', va=va,
            fontsize=6,  # 使用较小字体
            color=color,
            fontweight='bold' if color == 'white' else 'normal',
            rotation=90
        )
    
    # 设置子图标题（包含leiden和Celltype信息）
    # 缩短Celltype显示，只显示主要部分
    if len(celltype) > 20:
        # 如果Celltype太长，只显示前部分
        short_celltype = celltype # celltype[:17] + "..."
    else:
        short_celltype = celltype
    
    title_text =f'{short_celltype}'  # f'L{leiden}\n{short_celltype}'
    ax.set_title(title_text, fontsize=6, fontweight='bold', pad=3)
    
    # 隐藏x轴刻度
    ax.set_xticks([])
    
    # 设置y轴标签
    if i % n_cols == 0:  # 每行最左边的图显示y轴标签
        ax.set_ylabel('Sig Score', fontsize=8)
    
    # 设置y轴范围，为顶部标签留出空间
    y_max = grouped_mean[available_columns].values.max() * 1.15  # 增加15%的空间用于标签
    ax.set_ylim([0, y_max])
    
    # 添加网格线（仅y方向，更简洁）
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # 移除顶部和右侧边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 根据Celltype前缀添加背景色区分
    celltype_prefix =  celltype.split('_')[0]     # get_celltype_prefix(celltype) , 这个函数不在了
    if celltype_prefix == 'ArtEC':  # iFibro
        ax.set_facecolor('#FFF5E6')  # 浅橙色背景
    elif celltype_prefix == 'CapEC':  # eFibro
        ax.set_facecolor('#E6F7FF')  # 浅蓝色背景
    elif celltype_prefix == 'EndoMT':  # Myfibro
        ax.set_facecolor('#F0E6FF')  # 浅紫色背景
    # elif celltype_prefix == 'EndoMT-I':  # qFibro
    #     ax.set_facecolor('#E6FFE6')  # 浅绿色背景
    elif celltype_prefix == 'LEC':  # eFibro
        ax.set_facecolor("#E58606")  
    # elif celltype_prefix == 'Peri':  # Myfibro
    #     ax.set_facecolor("#BEBADA")  
    elif celltype_prefix == 'MCs':  # qFibro
        ax.set_facecolor("#9C9C5E")  
    elif celltype_prefix == 'VenEC':  # qFibro
        ax.set_facecolor("#F3D4F4")  # 

# 8. 隐藏多余的子图
for i in range(len(grouped_mean), len(axes)):
    axes[i].axis('off')

# 9. 创建统一的图例
# 使用简化版特征名称作为图例标签
# feature_short_names = []
# for col in available_columns:
#     if 'Tip' in col:
#         feature_short_names.append('Tip')
#         # feature_short_names.append('Myfibro_l')
#     elif 'AP' in col:
#         feature_short_names.append('AP')
#     elif 'Stalk' in col:
#         feature_short_names.append('Stalk')
feature_short_names = available_columns  # ['SMC', 'Peri', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT-I', 'EndoMT-II']


# 创建自定义的图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, color=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(available_columns))]


# 创建Celltype分组图例
celltype_prefix_colors = {
    1: ('ArtEC', '#FFF5E6'),
    2: ('CapEC', '#E6F7FF'),
    3: ('EndoMT', '#F0E6FF'),
   #  4: ('EndoMT-I', '#E6FFE6'),  
    4: ('LEC', '#E58606'),
   # 6: ('Peri', '#BEBADA'),
    5: ('MCs', '#9C9C5E'),
    6: ('VenEC', '#F3D4F4')
}

# 创建Celltype图例句柄
celltype_handles = []
celltype_labels = []

for prefix in sorted(celltype_prefix_colors.keys()):
    label, color = celltype_prefix_colors[prefix]
    handle = plt.Rectangle((0,0),1,1, color=color, edgecolor='gray', linewidth=0.5, alpha=0.7)
    celltype_handles.append(handle)
    celltype_labels.append(label)

########## 都是同一个图例，就用同一个图例就好，放在中间
# Subtypes图例（左下区域）
fig.legend(legend_handles, feature_short_names, 
           loc='lower center', 
           bbox_to_anchor=(0.25, 0.01),  # 底部左侧
           ncol=len(available_columns),  # 水平排列
           fontsize=8,
           title='marker_list',
           title_fontsize=9,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

# Celltype Groups图例（右下区域）
fig.legend(celltype_handles, celltype_labels,
           loc='lower center',
           bbox_to_anchor=(0.7, 0.01),  # 底部右侧
           ncol=8,  # 水平排列
           title='Celltype',
           title_fontsize=9,
           fontsize=8,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.11,left=0.05, right=0.9, hspace=0.4, wspace=0.3)

# 12. 保存图形
output_path = './TabulaStromal_figures/Endothelial//Endo_leiden_subtype_signature_scores_S.png'
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"图形已保存到: {output_path}")
# 13. 显示图形
plt.show()



In [ ]:
adata_pan.obs[['leiden','Celltype','Celltype_general']].drop_duplicates().sort_values('leiden')


In [ ]:
adata.obs[['leiden','Celltype','Celltype_general','Celltype_sub']].drop_duplicates().sort_values('leiden')


In [ ]:
############## 假如 leiden的cluster太多，用这种方式
import scanpy as sc
import matplotlib.pyplot as plt

# 假设你已经有了一个AnnData对象adata，并且已经计算了UMAP和Leiden聚类
# adata = ...

# 计算Leiden聚类
# sc.tl.leiden(adata, resolution=1.0)  # 你可以根据需要调整resolution参数

# 生成一个颜色映射，确保每种Leiden都有独特的颜色
n_clusters = len(adata.obs['GSE'].unique())


import numpy as np

num_colors = len(adata.obs['GSE'].unique())
colors = plt.cm.get_cmap("rainbow", num_colors)  # 可以换成其他 colormap，比如 "viridis", "rainbow"

# 转换为 RGB 颜色列表
color_list = [colors(i) for i in range(num_colors)]


sc.pl.umap(adata, color='GSE', palette=color_list)  # , legend_loc='on data'



In [ ]:
sorted(dir())


In [ ]:
def list_files(directory):
    file_list = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            file_list.append(file_path)
    return file_list
 

In [ ]:
Fib_marker_general = sorted(list(set(["COL1A1","COL1A2","COL3A1","COL6A1","IL6","MCAM","MMP2","MYL9", "MYLK","PDGFA"])))
Fib_marker_spec =sorted(list(set(['COL1A1','CFD','CTHRC1','POSTN','FAP','LRRC15','HAND2','SFRP1','IL6','CXCL3','APOD','PI16',
  'GPX1','IGF2','CCL19','SAA1','VWA1','CCL5','PTPRC','CCL4','CILP','THBS4','MYH11','MYLK','GJA4','ACTA2','RGS5'])))
Fib_marker_yilv = ['IGF2','ENG','FAP','PDGFRA','PECAM1','ITGB1','SPON1','MMP2','MYO1B','ARHGDIB','NDUFA4L2','HIGD1B','COX4I2','RGS5',
    'CNN1','DES','FGFR4','VEGFD','MME','LEFR','PI16','CD34','C3','CFD','C7','ADH1B','GREM1','CTHRC1','LRRC15','POSTN','MYH11','ACTA2','PDPN',
    'CXCL14','CXCL12','CCL2','COL12A1','COL11A1','COL10A1','COL8A1','COL5A1','COL4A2','COL4A1','COL3A1','COL1A2','COL1A1']
Fib_marker_union = list(set(Fib_marker_general) | set(Fib_marker_spec) | set(Fib_marker_yilv))
Endo_marker_general =["PECAM1","VWF" ,"CHGB",'CD144','CDH5','KDR','FLK1','CD54','CD105','CD62E','SELE']      # ["PECAM1","VWF" ,"CHGB"]
Endo_marker_spec = ['CSF3','NR2F2','TCIM','VCAM1','RGCC','APLN','ESM1','GJA4','BMX','CXCR4','PTPRC','CD3D','IGKC','CCL4','ACTA2','SFN','SPARC','PROX1','PDPN','CHGB']
Endo_marker_union = list(set(Endo_marker_general).union(set(Endo_marker_spec)))
MoMac_marker_general = ["CD14","CD68" ,"CSF1R","CST3","FABP4","MMP9","S100A9"] #
MoMac_marker_spec = ["FCN1","VCAN","S100A8","S100A9","CD14","FCGR3A","CX3CR1","CD68","MERTK","PPARG","MME","FABP4","MCEMP1","MARCO",
            "MRC1","CD163","CSF1R","GPNMB","APOE","SPP1","SELENOP","TREM2","MSR1","VEGFA","HIF1A","SERPINB9","TNF","CD86","CD40","HLA-DRA","HLA-DQA1","HLA-DPA1",
             "IL1B","CXCL3","CXCL8","CXCL9","CXCL10","CXCL11","CCL18","IDO1","CD274","SIRPA","ENTPD1","VSIR",
            "C1QC","C1QA","C3","CD55","CFP","SLC40A1","IL2RA","CCR2","FOLR2"]
MoMac_marker_union = list(set(MoMac_marker_general).union(set(MoMac_marker_spec)))


In [ ]:
adata_files = list_files('/fse/DC/mouse/qc_final_tmp/')   ## select the .h5ad files
adata_h5ad_files = []
for file in adata_files:
    if file.endswith('.h5ad'):
        adata_h5ad_files.append(file)
        

In [ ]:
# adata = sc.read_h5ad(adata_h5ad_files[0])
if 'leiden' in adata.obs.columns:
    

In [ ]:
#### 遍历每个 adata 文件，然后合并adata对象
adata_list = []
for file in adata_h5ad_files:
    adata_tmp = sc.read_h5ad(adata_h5ad_files)
    if 'leiden' in adata_tmp.obs.columns:
        adata_var_index_upper = [x.upper() for x in adata_tmp.var.index]  # mouse的需要做这一步
        Fib_union_gene_inter = set(adata_var_index_upper).intersection(Fib_marker_union)
        index_in_df = [adata_var_index_upper.index(x) for x in Fib_union_gene_inter]
        Fib_union_geneset = adata_tmp.var.iloc[index_in_df,].index.to_list()
        if len(Fib_union_geneset) > 0:
            sc.tl.score_genes(adata_tmp, Fib_union_geneset, score_name='MS_Fib_union', use_raw=False)  ### 没有这个函数  sc.tl.score_geneset
        ms_columns = [col for col in adata_tmp.obs.columns if 'MS_' in col]
        grouped_leiden_all = adata_tmp.obs.groupby('leiden')[ms_columns].describe()  ### adata
        # grouped_leiden_all = adata.obs.groupby('leiden')[['MS_Fib_general','MS_Fib_spec','MS_Fib_yilv','MS_Fib_union','MS_Endo_general','MS_Endo_spec','MS_Endo_union','MS_MoMac_general','MS_MoMac_spec','MS_MoMac_union']].describe()  ### adata.obs.groupby
        file.split('/')[3]
        thresh_tmp = 0.2
        ms_columns_fib = [item for item in ms_columns if 'Fib' in item]
        
         

In [ ]:
len(adata_h5ad_files)


In [ ]:
adata_tmp = sc.read_h5ad('/fse/DC/mouse/qc_final_tmp/GSE0nn/GSE54006/GSE54006_1_gene_count.h5ad')
if 'leiden' in adata_tmp.obs.columns:
    adata_var_index_upper = [x.upper() for x in adata_tmp.var.index]  # mouse的需要做这一步
    Fib_union_gene_inter = set(adata_var_index_upper).intersection(Fib_marker_union)
    index_in_df = [adata_var_index_upper.index(x) for x in Fib_union_gene_inter]
    Fib_union_geneset = adata_tmp.var.iloc[index_in_df,].index.to_list()
    if len(Fib_union_geneset) > 0:
        sc.tl.score_genes(adata_tmp, Fib_union_geneset, score_name='MS_Fib_union', use_raw=False)  ### 没有这个函数  sc.tl.score_geneset
    ms_columns = [col for col in adata_tmp.obs.columns if 'MS_' in col]
    grouped_leiden_all = adata_tmp.obs.groupby('leiden')[ms_columns].describe()  ### adata
    # grouped_leiden_all = adata.obs.groupby('leiden')[['MS_Fib_general','MS_Fib_spec','MS_Fib_yilv','MS_Fib_union','MS_Endo_general','MS_Endo_spec','MS_Endo_union','MS_MoMac_general','MS_MoMac_spec','MS_MoMac_union']].describe()  ### adata.obs.groupby
    file.split('/')[3]
    thresh_tmp = 0.2
    ms_columns_fib = [item for item in ms_columns if 'Fib' in item]
     

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import re
import os
Fib_summary_data = pd.read_csv('/mnt/Venus/home/wangyuting/Project/Aging/LLM_Aging/data/LLM_age_cluster/adata_h5/LLM_human_fib_filter_0.2_union.csv',sep=',')
Endo_summary_data = pd.read_csv('/mnt/Venus/home/wangyuting/Project/Aging/LLM_Aging/data/LLM_age_cluster/adata_h5/LLM_human_endo_filter_0.2_union.csv',sep=',')



In [ ]:
Fib_summary_data
Endo_summary_data


In [ ]:
Endo_summary_data

In [ ]:
Endo_summary_data['cell_num'].sum()

In [ ]:
Fib_summary_data['cell_num'].sum() 

In [ ]:
sorted(dir())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 绘制分布图
df = Fib_summary_data
plt.figure(figsize=(8, 6))
sns.histplot(df['clus_num'], kde=False)  # kde=True 添加核密度估计曲线

plt.title('Distribution of clus_num')
plt.xlabel('sample')
plt.ylabel('clus_num')

# 显示图形
plt.show()


In [ ]:
Fib_summary_data['clus_num'].max()

In [ ]:
Fib_summary_data[Fib_summary_data['clus_num']==100]

In [ ]:
Venus_unprocessed = list_files('/mnt/Saturn/home/wangyuting/Project/LLM_data/human/qc_final_tmp')


In [ ]:
import pickle
import os
import pandas as pd
from anndata import AnnData
with open('/mnt/Venus/home/wangyuting/Project/LLM/LLM_Aging/small_file/tisch_dataset_26.pkl','rb') as f:
    data_list = pickle.load(f)
    

In [ ]:
import sys
import scanpy as sc
import anndata as ad

print("Python version:", sys.version)
print("Scanpy version:", sc.__version__)
print("AnnData version:", ad.__version__)


In [ ]:
# os.getcwd()

In [ ]:

################################ Fibroblast, TISCH

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import random

DefaulfColorPalette = [
    "#E58606", "#5D69B1", "#52BCA3", "#99C945", "#CC61B0", "#24796C",
    "#DAA51B", "#2F8AC4", "#764E9F", "#ED645A", "#A5AA99", "#BCBD22",
    "#B279A2", "#EECA3B", "#17BECF", "#FF9DA6", "#778AAE", "#1B9E77",
    "#A6761D", "#526A83", "#B82E2E", "#80B1D3", "#68855C", "#D95F02",
    "#BEBADA", "#AF6458", "#D9AF6B", "#9C9C5E", "#625377", "#8C785D",
    "#88CCEE", "#E73F74", "#FFFFB3", "#CCEBC5", "#332288", "#A65628"
]

# 假设 data_list 是包含 4 个单细胞数据集的信息，形式如下：
# 每个数据集是一个字典，包含 'recall_df' (DataFrame) 和 'adata' (AnnData)
select_index = [3,5,6,8] 
adata_list_new = [data_list[i] for i in select_index]
# 创建一个 3 行 4 列的画布
fig, axes = plt.subplots(3, 4, figsize=(24, 14))

# 遍历每个数据集
for col, data in enumerate(adata_list_new):
    # 第一行：画 Recall 折线图
    recall_df = data['recall_df'][data['recall_df']['Celltype']=='Fibroblasts']
    # print(recall_df)
    ax = axes[0, col]
    # for column in recall_df.columns[1:]:
    for marker in recall_df['Marker'].unique():
        marker_data = recall_df[recall_df['Marker'] == marker]
        ax.plot(marker_data['thresh'], marker_data['Recall'], label=marker, marker='o')
        # ax.plot(recall_df['thresh'], recall_df['Recall'], label='Marker', marker='o')
        ax.set_ylim(0,1.1)
        ax.set_title(f"Recall of "+ list(recall_df['dataset'].unique())[0])
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Recall")
        ax.legend()

    # 第二行：UMAP (cell types)
    adata = data['adata']
    existing_celltypes = adata.obs['Celltype_curated'].unique()
    palette = [Palette_dict[ct] for ct in existing_celltypes if ct in Palette_dict]
    sc.pl.umap(adata, color="Celltype_curated", ax=axes[1, col],palette = palette , show=False)
    axes[1, col].set_title(f"Celltype of "+ list(recall_df['dataset'].unique())[0])

    # 第三行：UMAP (source)
    sc.pl.umap(adata, color="MS_Fib_union", ax=axes[2, col], show=False)
    axes[2, col].set_title(f"Fib score of "+list(recall_df['dataset'].unique())[0])
    
    # 第四行，预测的celltype 
    # adata.obs['Fib_pred'] = 'None'
    # adata.obs[adata.obs['leiden'].isin(index_Fib_general)]['Fib_pred'] ='Fibroblast'
    # sc.pl.umap(adata, color="Fib_pred", ax=axes[3, col], show=False)
    # axes[2, col].set_title(f"Fib predict of "+list(recall_df['dataset'].unique())[0])
    
# 调整布局
plt.tight_layout()
plt.savefig('./small_file/tisch_fibro_0.2_union.pdf',format='pdf', bbox_inches='tight')
plt.show()


In [ ]:

################################ Endothelial, TISCH

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

DefaulfColorPalette = [
    "#E58606", "#5D69B1", "#52BCA3", "#99C945", "#CC61B0", "#24796C",
    "#DAA51B", "#2F8AC4", "#764E9F", "#ED645A", "#A5AA99", "#BCBD22",
    "#B279A2", "#EECA3B", "#17BECF", "#FF9DA6", "#778AAE", "#1B9E77",
    "#A6761D", "#526A83", "#B82E2E", "#80B1D3", "#68855C", "#D95F02",
    "#BEBADA", "#AF6458", "#D9AF6B", "#9C9C5E", "#625377", "#8C785D",
    "#88CCEE", "#E73F74", "#FFFFB3", "#CCEBC5", "#332288", "#A65628"
]

# 假设 data_list 是包含 4 个单细胞数据集的信息，形式如下：
# 每个数据集是一个字典，包含 'recall_df' (DataFrame) 和 'adata' (AnnData)

select_index = [0,2,3,4]
adata_list_new = [data_list[i] for i in select_index]
# 创建一个 3 行 4 列的画布
fig, axes = plt.subplots(3, 4, figsize=(24, 14))

# 遍历每个数据集
for col, data in enumerate(adata_list_new):
    # 第一行：画 Recall 折线图
    recall_df = data['recall_df'][data['recall_df']['Celltype']=='Endothelial']
    # print(recall_df)
    ax = axes[0, col]
    # for column in recall_df.columns[1:]:
    for marker in recall_df['Marker'].unique():
        marker_data = recall_df[recall_df['Marker'] == marker]
        ax.plot(marker_data['thresh'], marker_data['Recall'], label=marker, marker='o')
        # ax.plot(recall_df['thresh'], recall_df['Recall'], label='Marker', marker='o')
        ax.set_ylim(0,1.1)
        ax.set_title(f"Recall of "+ list(recall_df['dataset'].unique())[0])
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Recall")
        ax.legend()

    # 第二行：UMAP (cell types)
    adata = data['adata']
    existing_celltypes = adata.obs['Celltype_curated'].unique()
    palette = [Palette_dict[ct] for ct in existing_celltypes if ct in Palette_dict]
    sc.pl.umap(adata, color="Celltype_curated", ax=axes[1, col],palette = palette, show=False)
    axes[1, col].set_title(f"Celltype of "+ list(recall_df['dataset'].unique())[0])

    # 第三行：UMAP (source)
    sc.pl.umap(adata, color="MS_Endo_union", ax=axes[2, col], show=False)
    axes[2, col].set_title(f"Endo score of "+list(recall_df['dataset'].unique())[0])
    
    # 第四行，预测的celltype 
    # adata.obs['Fib_pred'] = 'None'
    # adata.obs[adata.obs['leiden'].isin(index_Fib_general)]['Fib_pred'] ='Fibroblast'
    # sc.pl.umap(adata, color="Fib_pred", ax=axes[3, col], show=False)
    # axes[2, col].set_title(f"Fib predict of "+list(recall_df['dataset'].unique())[0])
    
# 调整布局
plt.tight_layout()
plt.savefig('./small_file/tisch_endothelial_0.2_union.pdf',format='pdf', bbox_inches='tight')
plt.show()


In [ ]:

################################ Mono/Macro, TISCH

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

DefaulfColorPalette = [
    "#E58606", "#5D69B1", "#52BCA3", "#99C945", "#CC61B0", "#24796C",
    "#DAA51B", "#2F8AC4", "#764E9F", "#ED645A", "#A5AA99", "#BCBD22",
    "#B279A2", "#EECA3B", "#17BECF", "#FF9DA6", "#778AAE", "#1B9E77",
    "#A6761D", "#526A83", "#B82E2E", "#80B1D3", "#68855C", "#D95F02",
    "#BEBADA", "#AF6458", "#D9AF6B", "#9C9C5E", "#625377", "#8C785D",
    "#88CCEE", "#E73F74", "#FFFFB3", "#CCEBC5", "#332288", "#A65628"
]

# 假设 data_list 是包含 4 个单细胞数据集的信息，形式如下：
# 每个数据集是一个字典，包含 'recall_df' (DataFrame) 和 'adata' (AnnData)

select_index = [0,1,2,3]
adata_list_new = [data_list[i] for i in select_index]
# 创建一个 3 行 4 列的画布
fig, axes = plt.subplots(3, 4, figsize=(24, 14))
  
# 遍历每个数据集
for col, data in enumerate(adata_list_new):
    # 第一行：画 Recall 折线图
    recall_df = data['recall_df'][data['recall_df']['Celltype']=='Mono/Macro']
    # print(recall_df)
    ax = axes[0, col]
    # for column in recall_df.columns[1:]:
    for marker in recall_df['Marker'].unique():
        marker_data = recall_df[recall_df['Marker'] == marker]
        ax.plot(marker_data['thresh'], marker_data['Recall'], label=marker, marker='o')
        # ax.plot(recall_df['thresh'], recall_df['Recall'], label='Marker', marker='o')
        ax.set_ylim(0,1.1)
        ax.set_title(f"Recall of "+ list(recall_df['dataset'].unique())[0])
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Recall")
        ax.legend()

    # 第二行：UMAP (cell types)
    adata = data['adata']
    existing_celltypes = adata.obs['Celltype_curated'].unique()
    palette = [Palette_dict[ct] for ct in existing_celltypes if ct in Palette_dict]
    sc.pl.umap(adata, color="Celltype_curated", ax=axes[1, col],palette = palette, show=False)
    axes[1, col].set_title(f"Celltype of "+ list(recall_df['dataset'].unique())[0])

    # 第三行：UMAP (source)
    sc.pl.umap(adata, color="MS_MoMac_union", ax=axes[2, col], show=False)
    axes[2, col].set_title(f"Momac score of "+list(recall_df['dataset'].unique())[0])
    
    # 第四行，预测的celltype 
    # adata.obs['Fib_pred'] = 'None'
    # adata.obs[adata.obs['leiden'].isin(index_Fib_general)]['Fib_pred'] ='Fibroblast'
    # sc.pl.umap(adata, color="Fib_pred", ax=axes[3, col], show=False)
    # axes[2, col].set_title(f"Fib predict of "+list(recall_df['dataset'].unique())[0])
    
# 调整布局
plt.tight_layout()
plt.savefig('./small_file/tisch_monomacro_0.2_union.pdf',format='pdf', bbox_inches='tight')
plt.show()


In [ ]:
############################## 画proportion的图
import matplotlib.pyplot as plt
import pandas as pd

# 假设data_list是一个包含字典的列表，其中每个字典都有"proportion"这个key，对应一个DataFrame
# 示例：data_list[0]["proportion"] 是一个包含 celltype 和 proportion 的DataFrame

# 创建画布
fig, axes = plt.subplots(3, 9, figsize=(30, 13))  # 4行5列
axes = axes.flatten()  # 将axes二维数组转为一维，方便迭代

# 遍历data_list并绘制柱状图
for i, data in enumerate(data_list):
    ax = axes[i]
    proportion_df = data["proportion"]  # 获取proportion DataFrame
    dataset = data['recall_df']['dataset'].unique()[0]
    # 将DataFrame转换为适合绘图的格式
    pivot_df = proportion_df.pivot(index='celltype', columns='type', values='proportion')
    pivot_df.plot(kind='bar', ax=ax, color=['skyblue', 'orange'], legend=False)  # 绘制柱状图
    
    # 设置每张图的标题
    ax.set_title(dataset)
    ax.set_xlabel("Cell Type")
    ax.set_ylabel("Proportion")
    ax.set_ylim(0,0.7)
    # 显示图例在右上角
    ax.legend(["Curated", "Predicted"], loc='upper right')

# 调整布局避免重叠
plt.tight_layout()
plt.savefig('./small_file/tisch_fraction_0.2_union.pdf',format='pdf', bbox_inches='tight')
plt.show()


In [ ]:
len(data_list)  ### TISCH中需要check的 datasets
Palette_dict


In [ ]:
len(data_list)


In [ ]:
data_list[0]


In [ ]:
DefaulfColorPalette = [
    "#E58606", "#5D69B1", "#52BCA3", "#99C945", "#CC61B0", "#24796C",
    "#DAA51B", "#2F8AC4", "#764E9F", "#ED645A", "#A5AA99", "#BCBD22",
    "#B279A2", "#EECA3B", "#17BECF", "#FF9DA6", "#778AAE", "#1B9E77",
    "#A6761D", "#526A83", "#B82E2E", "#80B1D3", "#68855C", "#D95F02",
    "#BEBADA", "#AF6458", "#D9AF6B", "#9C9C5E", "#625377", "#8C785D",
    "#88CCEE", "#E73F74", "#FFFFB3", "#CCEBC5", "#332288", "#A65628"
]
import matplotlib.pyplot as plt
import colorsys
def generate_100_colors():
    num_colors = 100
    colors = []
    for i in range(num_colors):
        # 使用 HSL 色彩空间均匀分布颜色
        hue = i / num_colors  # 色相 (0-1)，通过步长确保均匀
        saturation = 0.65     # 饱和度，固定值
        lightness = 0.55      # 亮度，固定值
        rgb = colorsys.hls_to_rgb(hue, lightness, saturation)  # 转换为 RGB
        # 转换为十六进制颜色格式
        hex_color = '#%02X%02X%02X' % (int(rgb[0] * 255), int(rgb[1] * 255), int(rgb[2] * 255))
        colors.append(hex_color)
    return colors

# 生成颜色列表
color_list = generate_100_colors() 

In [ ]:
ct_unique_all = []
for i in range(len(data_list)):
    adata_tmp = data_list[i]['adata']
    ct_unique = sorted(list(adata_tmp.obs['Celltype_curated'].unique()))
    ct_unique_all = ct_unique_all + ct_unique

ct_unique_all = sorted(list(set(ct_unique_all)))


In [ ]:
color_list = generate_100_colors()  # 需要多少种颜色就产生多少种颜色
color_list.remove('#4147D6')  ## 先删除掉 3种颜色，后面复制给3个lineage
color_list.remove('#D64141')
color_list.remove('#A7D641')
Palette_dict = {k: v for k, v in zip(ct_unique_all, color_list)}
Palette_dict['Fibroblasts'] = '#D64141'
Palette_dict['Endothelial'] = '#A7D641'
Palette_dict['Mono/Macro'] = '#4147D6'


In [ ]:
len(Palette_dict)


In [ ]:
sorted(dir())



In [ ]:
sorted(dir())


In [ ]:
######### 统计不同lineage中tissue 和 cell_number fraction的分布
Endo_human_meta = pd.read_csv('./data/scalex/endo/human/Endo_fst_human_just_filter_meta_df_1.tsv',sep='\t')
Fib_human_meta = pd.read_csv('./data/scalex/fibro/human/Fibro_fst_human_just_filter_meta_df_1.tsv',sep='\t')
Momac_human_meta = pd.read_csv('./data/scalex/momac/human/Momac_fst_human_just_filter_meta_df.tsv',sep='\t')
gse_gsm_sample_human = pd.read_csv('/fse/DC/human/qc_final_tmp/reserve.txt',sep=',',header=None)




In [ ]:
Fib_human_cell_num = pd.merge(Fib_human_meta, gse_gsm_sample_human, on=['gse','gsm'],how='left')
Fib_human_cell_num_blood_sub = Fib_human_cell_num[Fib_human_cell_num['tissue_final']=='blood'][['gse','gsm','cell_num','cell1','qc_cell_num']]
Fib_human_cell_num_blood_sub

In [ ]:
Fib_human_cell_num_blood_sub.drop_duplicates()

In [ ]:
gse_gsm_sample_human['gse'] = gse_gsm_sample_human[0].str.split("/").str.get(6)
gse_gsm_sample_human['gsm'] = gse_gsm_sample_human[0].str.split("/").str.get(7).str.split("_").str.get(0)
gse_gsm_sample_human.columns = ['path','cell1','qc_cell_num','gse','gsm']


In [ ]:
# gse_gsm_sample_human.columns = ['path','cell1','qc_cell_num','gse','gsm']

help(pd.merge)


In [ ]:
Endo_human_cell_num = pd.merge(Endo_human_meta, gse_gsm_sample_human, on=['gse','gsm'],how='left')
Endo_human_cell_num_blood_sub = Endo_human_cell_num[Endo_human_cell_num['tissue_final']=='blood'][['gse','gsm','cell_num','cell1','qc_cell_num']]
# Endo_human_cell_num_blood_sub = Endo_human_cell_num_blood_sub[Endo_human_cell_num_blood_sub['fraction'] <= 0.8]
Endo_human_cell_num_blood_sub

In [ ]:
Fib_human_cell_num_blood_sub.drop_duplicates()


In [ ]:
Fib_human_cell_num_blood_sub['cell_num'].sum()


In [ ]:
df_sorted

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Fib_human_cell_num_blood_sub['fraction'] = Fib_human_cell_num_blood_sub['cell_num'] / Fib_human_cell_num_blood_sub['qc_cell_num']
Fib_human_cell_num_blood_sub = Fib_human_cell_num_blood_sub[Fib_human_cell_num_blood_sub['fraction'] <= 0.8]
# 按比例排序
df_sorted = Fib_human_cell_num_blood_sub.sort_values(by='fraction', ascending=False).reset_index(drop=True)

# 绘制柱状图
plt.figure(figsize=(10, 6))
plt.bar(range(1, len(df_sorted) + 1), df_sorted['fraction'], color='dodgerblue') # , edgecolor='black'

# 添加标题和标签
plt.title('blood cell fraction', fontsize=16)
plt.xlabel('Index', fontsize=12)
plt.ylabel('Fraction', fontsize=12)

# 优化布局
plt.tight_layout()

# 显示图表
plt.show()


In [ ]:
Momac_human_meta_sub = Momac_human_meta[['gse','gsm','tissue_final']]
Momac_human_meta_sub

In [ ]:
sorted(dir())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 将 tissue_final 列拆分为独立的组织并展开
# tissue_split = df['tissue_final'].str.split(';', expand=True).stack().reset_index(drop=True)

# 统计每个组织的出现次数
tissue_counts = Momac_human_meta_sub['tissue_final'].value_counts()

# 选取前 20 个组织
top_20_tissues = tissue_counts.head(20)

plt.figure(figsize=(12, 6))
bars = top_20_tissues.sort_values(ascending=False).plot(kind='bar', color='skyblue', edgecolor='black')

# 添加柱子顶部的数值
for i, value in enumerate(top_20_tissues.sort_values(ascending=False)):
    plt.text(i, value + 0.5, str(value), ha='center', fontsize=10)

# 添加标题和轴标签
plt.title('Top 20 Tissue Types by Frequency', fontsize=16)
plt.xlabel('Tissue Type', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

# 调整标签显示
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()

# 显示图表
plt.show()


In [ ]:
Endo_human_meta_sub = Endo_human_meta[['gse','gsm','tissue_final']]
#Endo_human_meta_sub

In [ ]:
# tissue_split = Endo_human_meta['tissue_final'].str.split(';', expand=True).stack().reset_index(drop=True)
# tissue_split
Endo_human_meta_cell_sub['tissue_final']

In [ ]:
Endo_human_cell_num # Endo_human_meta, 

In [ ]:
Endo_human_meta

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt



# 将 tissue_final 列拆分为独立的组织并展开
# tissue_split = df['tissue_final'].str.split(';', expand=True).stack().reset_index(drop=True)

# 统计每个组织的出现次数
Fib_human_cell_num['fraction'] = Fib_human_cell_num['cell_num'] / Fib_human_cell_num['qc_cell_num']
Fib_human_cell_num = Fib_human_cell_num[Fib_human_cell_num['fraction']<0.8]
Fib_human_meta_sub = Fib_human_cell_num[['gse','gsm','tissue_final']]
tissue_counts = Fib_human_meta_sub['tissue_final'].value_counts()

# 选取前 20 个组织
top_20_tissues = tissue_counts.head(20)

plt.figure(figsize=(12, 6))
bars = top_20_tissues.sort_values(ascending=False).plot(kind='bar', color='skyblue', edgecolor='black')

# 添加柱子顶部的数值
for i, value in enumerate(top_20_tissues.sort_values(ascending=False)):
    plt.text(i, value + 0.5, str(value), ha='center', fontsize=10)

# 添加标题和轴标签
plt.title('Top 20 Tissue Types by Frequency', fontsize=16)
plt.xlabel('Tissue Type', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

# 调整标签显示
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()

# 显示图表
plt.show()

In [ ]:
Endo_human_meta_cell_sub = Endo_human_cell_num[['gse','gsm','tissue_final','cell_num']]
Endo_human_meta_cell_sub.reset_index(drop=True, inplace=True)
Endo_human_meta_cell_sub

In [ ]:
Fib_human_meta_cell_sub = Fib_human_cell_num[['gse','gsm','tissue_final','cell_num']]
Fib_human_meta_cell_sub.reset_index(drop=True, inplace=True)
Fib_human_meta_cell_sub

In [ ]:
Endo_human_cell_num


In [ ]:
Endo_human_cell_num['fraction'] = Endo_human_cell_num['cell_num'] / Endo_human_cell_num['qc_cell_num']
Endo_human_cell_num = Endo_human_cell_num[Endo_human_cell_num['fraction']<=0.8]
Endo_human_meta_sub = Endo_human_cell_num[['gse','gsm','tissue_final']]


In [ ]:
Endo_human_cell_num

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# 将 tissue_final 列拆分为独立的组织并展开
# tissue_split = df['tissue_final'].str.split(';', expand=True).stack().reset_index(drop=True)

# 重新构建 DataFrame，将拆分后的组织与 cell_num 对应
tissue_expanded = pd.DataFrame({
    'tissue_final': Fib_human_meta_cell_sub['tissue_final'],
    'cell_num': Fib_human_meta_cell_sub.loc[Fib_human_meta_cell_sub['tissue_final'].index // len(Fib_human_meta_cell_sub.columns), 'cell_num'].values
})

# 按 tissue_final 分组并计算 cell_num 的总和
tissue_sums = tissue_expanded.groupby('tissue_final')['cell_num'].sum()

# 判断是否需要对 cell_num 总和取 log10 转换
if tissue_sums.max() / tissue_sums.min() > 100:  # 如果最大值与最小值相差超过100倍
    tissue_sums_log = np.log10(tissue_sums)  # 进行 log10 转换
    log_transformed = True
else:
    tissue_sums_log = tissue_sums
    log_transformed = False

# 选取前 20 个 tissue_final
top_20_tissues = tissue_sums_log.sort_values(ascending=False).head(20)
top_20_original_values = tissue_sums[top_20_tissues.index]  # 对应的原始值

# 绘制柱状图
plt.figure(figsize=(12, 6))
bars = top_20_tissues.sort_values(ascending=False).plot(kind='bar', color='skyblue', edgecolor='black')

# 添加柱子顶部的原始 cell_num 总和数值
for i, value in enumerate(top_20_original_values):
    plt.text(i, top_20_tissues[i] + 0.1, str(value), ha='center', fontsize=10)

# 添加标题和轴标签
plt.title('Top 20 tissue by cell num', fontsize=16)
if log_transformed:
    plt.ylabel('Total Cell Number(log10)', fontsize=12)
else:
    plt.ylabel('Total Cell Number', fontsize=12)
plt.xlabel('Tissue Type', fontsize=12)

# 调整标签显示
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()

# 显示图表
plt.show()


In [ ]:
Endo_human_meta_cell_sub = Momac_human_meta[['gse','gsm','tissue_final','cell_num']]
Endo_human_meta_cell_sub

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# 将 tissue_final 列拆分为独立的组织并展开
# tissue_split = df['tissue_final'].str.split(';', expand=True).stack().reset_index(drop=True)

# 重新构建 DataFrame，将拆分后的组织与 cell_num 对应
tissue_expanded = pd.DataFrame({
    'tissue_final': Endo_human_meta_cell_sub['tissue_final'],
    'cell_num': Endo_human_meta_cell_sub.loc[Endo_human_meta_cell_sub['tissue_final'].index // len(Endo_human_meta_cell_sub.columns), 'cell_num'].values
})

# 按 tissue_final 分组并计算 cell_num 的总和
tissue_sums = tissue_expanded.groupby('tissue_final')['cell_num'].sum()

# 判断是否需要对 cell_num 总和取 log10 转换
if tissue_sums.max() / tissue_sums.min() > 100:  # 如果最大值与最小值相差超过100倍
    tissue_sums_log = np.log10(tissue_sums)  # 进行 log10 转换
    log_transformed = True
else:
    tissue_sums_log = tissue_sums
    log_transformed = False

# 选取前 20 个 tissue_final
top_20_tissues = tissue_sums_log.sort_values(ascending=False).head(20)
top_20_original_values = tissue_sums[top_20_tissues.index]  # 对应的原始值

# 绘制柱状图
plt.figure(figsize=(12, 6))
bars = top_20_tissues.sort_values(ascending=False).plot(kind='bar', color='skyblue', edgecolor='black')

# 添加柱子顶部的原始 cell_num 总和数值
for i, value in enumerate(top_20_original_values):
    plt.text(i, top_20_tissues[i] + 0.1, str(value), ha='center', fontsize=10)

# 添加标题和轴标签
plt.title('Top 20 tissue by cell num', fontsize=16)
if log_transformed:
    plt.ylabel('Total Cell Number(log10)', fontsize=12)
else:
    plt.ylabel('Total Cell Number', fontsize=12)
plt.xlabel('Tissue Type', fontsize=12)

# 调整标签显示
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()

# 显示图表
plt.show()


In [ ]:
Fib_marker_general = sorted(list(set(["COL1A1","COL1A2","COL3A1","COL6A1","IL6","MCAM","MMP2","MYL9", "MYLK","PDGFA"])))
Fib_marker_spec =sorted(list(set(['COL1A1','CFD','CTHRC1','POSTN','FAP','LRRC15','HAND2','SFRP1','IL6','CXCL3','APOD','PI16',
  'GPX1','IGF2','CCL19','SAA1','VWA1','CCL5','CCL4','CILP','THBS4','MYH11','MYLK','GJA4','ACTA2','RGS5'])))  # 'PTPRC'
Fib_marker_yilv = ['IGF2','ENG','FAP','PDGFRA','PECAM1','ITGB1','SPON1','MMP2','MYO1B','ARHGDIB','NDUFA4L2','HIGD1B','COX4I2','RGS5',
    'CNN1','DES','FGFR4','VEGFD','MME','LEFR','PI16','C3','CFD','C7','ADH1B','GREM1','CTHRC1','LRRC15','POSTN','MYH11','ACTA2','PDPN',
    'CXCL14','CXCL12','CCL2','COL12A1','COL11A1','COL10A1','COL8A1','COL5A1','COL4A2','COL4A1','COL3A1','COL1A2','COL1A1'] # ,'CD34'
Fib_marker_union = list(set(Fib_marker_general) | set(Fib_marker_spec) | set(Fib_marker_yilv))
Endo_marker_general =["PECAM1","VWF" ,"CHGB",'CD144','CDH5','KDR','FLK1','CD54','CD105','CD62E','SELE']      # ["PECAM1","VWF" ,"CHGB"]
Endo_marker_spec = ['NR2F2','TCIM','VCAM1','RGCC','APLN','ESM1','BMX','CXCR4','IGKC','SFN','SPARC','PROX1','PDPN','CHGB']  # 'PTPRC','ACTA2','CCL4','GJA4'
Endo_marker_union = list(set(Endo_marker_general).union(set(Endo_marker_spec)))
MoMac_marker_general = ["CD14","CD68" ,"CSF1R","CST3","FABP4","MMP9","S100A9"] #
MoMac_marker_spec = ["FCN1","VCAN","S100A8","S100A9","CD14","FCGR3A","CX3CR1","CD68","MERTK","PPARG","MME","FABP4","MCEMP1","MARCO",
            "MRC1","CD163","CSF1R","GPNMB","APOE","SPP1","SELENOP","TREM2","MSR1","VEGFA","HIF1A","SERPINB9","TNF","CD86","CD40","HLA-DRA","HLA-DQA1","HLA-DPA1",
             "IL1B","CXCL3","CXCL8","CXCL9","CXCL10","CXCL11","CCL18","IDO1","CD274","SIRPA","ENTPD1","VSIR",
            "C1QC","C1QA","C3","CD55","CFP","SLC40A1","IL2RA","CCR2","FOLR2",'CSF3']
MoMac_marker_union = list(set(MoMac_marker_general).union(set(MoMac_marker_spec)))

In [ ]:
import scanpy as sc
adata= sc.read_h5ad('/fse/DC/human/qc_final_tmp/GSE154/GSE154386/GSM4670213_1_gene_count.h5ad')
adata


In [ ]:
sc.pl.umap(adata, color="leiden", show=False)


In [ ]:
# 可视化每个Leiden聚类的前10个差异表达基因
# sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False, groupby='leiden')


In [ ]:
import scanpy as sc
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# 确保所有 Endo_marker 都在 adata.var_names 中

# all_marker_genes = 
common_genes = [gene for gene in Fib_marker_union if gene in adata.var_names]

# 检查是否有共同基因
if not common_genes:
    print("No common genes found between Endo_marker and adata.var_names.")
else:
    # 提取表达矩阵并添加 Leiden 群注释
    adata_subset = adata[:, common_genes].to_df()  # 提取 Endo_marker 基因表达数据
    adata_subset['leiden'] = adata.obs['leiden'].values  # 添加 Leiden 注释

    # 计算每个基因在每个 Leiden 群中的平均表达
    grouped_data = adata_subset.groupby('leiden').mean()

    # 转换数据用于可视化
    melted_data = grouped_data.reset_index().melt(id_vars='leiden', var_name='Gene', value_name='Expression')

    # 绘制热图展示基因表达情况
    plt.figure(figsize=(12, 8))
    sns.heatmap(
        grouped_data.T,
        cmap="viridis",
        xticklabels=grouped_data.index,
        yticklabels=grouped_data.columns,
        cbar_kws={'label': 'Average Expression'}
    )
    plt.xlabel("Leiden ")
    plt.ylabel("Fibro_marker Genes")
    plt.title("Ave Expr of Fibro_marker Genes")
    plt.tight_layout()
    plt.savefig('external_inputs/LLM_Endo_blood_check_FibMarker_sample1_exp.pdf',format='pdf', bbox_inches='tight')
    # 显示图表
    plt.show()

In [ ]:
import os
os.getcwd()

In [ ]:
sorted(dir())


In [ ]:
Endo_human_meta


In [ ]:
Endo_human_cell_num_blood_sub_new = pd.merge(Endo_human_cell_num_blood_sub, gse_gsm_sample_human, on=['gse','gsm'],how='left')
Endo_human_cell_num_blood_sub_new_1 = Endo_human_cell_num_blood_sub_new[Endo_human_cell_num_blood_sub_new['fraction']>0.8]
Endo_human_cell_num_blood_sub_new_1

In [ ]:
need_check_path =  sorted(list(Endo_human_cell_num_blood_sub_new_1['path'].unique()))
need_check_path = [file.replace('.h5','.h5ad') for file in need_check_path]
need_check_path = [file.replace('/raw_mat_meta/','/qc_final_tmp/') for file in need_check_path]


In [ ]:

for path in need_check_path:
    print(path)
    adata_tmp = sc.read_h5ad(path)
    

In [ ]:
path = need_check_path[5]
adata_tmp = sc.read_h5ad(path)
adata_tmp

In [ ]:
reserve_samples = reserve_samples = pd.read_csv('external_inputs/reserve_sample_sex.csv',sep=',')
tissue_rectify_df = pd.read_csv('external_inputs/tissue_meta_rectify_1.csv',sep=',')
# tissue_rectify_df = pd.read_csv('external_inputs/tissue_meta_rectify_with_none.csv',sep=',')
# platform_rectify_df = pd.read_csv('external_inputs/platform_meta_rectify.csv',sep=',')
platform_rectify_df = pd.read_csv('external_inputs/platform_meta_rectify_1.csv',sep=',')  ### 都没有把 None 和 'none'的读取进来
treatment_rectify_df = pd.read_csv('external_inputs/treatment_meta_rectify_1.csv',sep=',')
race_rectify_df = pd.read_csv('external_inputs/race_meta_rectify_1.csv',sep=',')
sex_rectify_df = pd.read_csv('external_inputs/sex_meta_rectify_1.csv',sep=',')
age_rectify_df = pd.read_csv('external_inputs/age_meta_rectify_1.csv',sep=',')
disease_rectify_df = pd.read_csv('external_inputs/disease_meta_rectify_1.csv',sep=',')


In [ ]:
sc.pl.umap(adata_tmp, color="leiden",  show=False)


In [ ]:
list(adata_tmp.var_names)[0:7]


In [ ]:
Fib_marker_general = sorted(list(set(["COL1A1","COL1A2","COL3A1","COL6A1","IL6","MCAM","MMP2","MYL9", "MYLK","PDGFA"])))
Fib_marker_spec =sorted(list(set(['COL1A1','CFD','CTHRC1','POSTN','FAP','LRRC15','HAND2','SFRP1','IL6','CXCL3','APOD','PI16',
  'GPX1','IGF2','CCL19','SAA1','VWA1','CCL5','PTPRC','CCL4','CILP','THBS4','MYH11','MYLK','GJA4','ACTA2','RGS5'])))
Fib_marker_yilv = ['IGF2','ENG','FAP','PDGFRA','PECAM1','ITGB1','SPON1','MMP2','MYO1B','ARHGDIB','NDUFA4L2','HIGD1B','COX4I2','RGS5',
    'CNN1','DES','FGFR4','VEGFD','MME','LEFR','PI16','CD34','C3','CFD','C7','ADH1B','GREM1','CTHRC1','LRRC15','POSTN','MYH11','ACTA2','PDPN',
    'CXCL14','CXCL12','CCL2','COL12A1','COL11A1','COL10A1','COL8A1','COL5A1','COL4A2','COL4A1','COL3A1','COL1A2','COL1A1']
Fib_marker_union = list(set(Fib_marker_general) | set(Fib_marker_spec) | set(Fib_marker_yilv))
Endo_marker_general =["PECAM1","VWF" ,"CHGB",'CD144','CDH5','KDR','FLK1','CD54','CD105','CD62E','SELE']      # ["PECAM1","VWF" ,"CHGB"]
Endo_marker_spec = ['CSF3','NR2F2','TCIM','VCAM1','RGCC','APLN','ESM1','GJA4','BMX','CXCR4','PTPRC','CD3D','IGKC','CCL4','ACTA2','SFN','SPARC','PROX1','PDPN','CHGB']
Endo_marker_union = list(set(Endo_marker_general).union(set(Endo_marker_spec)))
MoMac_marker_general = ["CD14","CD68" ,"CSF1R","CST3","FABP4","MMP9","S100A9"] #
MoMac_marker_spec = ["FCN1","VCAN","S100A8","S100A9","CD14","FCGR3A","CX3CR1","CD68","MERTK","PPARG","MME","FABP4","MCEMP1","MARCO",
            "MRC1","CD163","CSF1R","GPNMB","APOE","SPP1","SELENOP","TREM2","MSR1","VEGFA","HIF1A","SERPINB9","TNF","CD86","CD40","HLA-DRA","HLA-DQA1","HLA-DPA1",
             "IL1B","CXCL3","CXCL8","CXCL9","CXCL10","CXCL11","CCL18","IDO1","CD274","SIRPA","ENTPD1","VSIR",
            "C1QC","C1QA","C3","CD55","CFP","SLC40A1","IL2RA","CCR2","FOLR2"]
MoMac_marker_union = list(set(MoMac_marker_general).union(set(MoMac_marker_spec)))


In [ ]:
import scanpy as sc
import pandas as pd
# 计算每个cluster的marker基因
# sc.tl.mark_groups(adata_tmp, groupby='leiden', keys=adata_tmp.obs['leiden'].unique())
# adata_tmp = adata_tmp.raw.to_adata()  # 
# 绘制dotplot图
sc.pl.dotplot(adata_tmp, var_names=list(set(list(adata_tmp.var_names)) & set(Endo_marker_union)) , groupby='leiden', dendrogram=False)


In [ ]:
help(set.intersection)

In [ ]:
adata_list_merge =[]
adata_summary_df = pd.DataFrame(columns=['gse','gsm','lineage','cell_num','clus_num'])
for data_index in range(len(need_check_path)):  ###   range(len(need_check_path))
    os.chdir('/fse/DC/human/qc_final_tmp/')
    dataset_file = need_check_path[data_index]
    print(dataset_file)
    adata_tmp = sc.read_h5ad(dataset_file)
    adata_tmp.obs['GSE'] = dataset_file.split('/')[6]
    gsm_prefix = dataset_file.split('/')[7]
    adata_tmp.obs['GSM'] = gsm_prefix.split('_')[0]
    age_tmp = age_rectify_df[(age_rectify_df['gse']==adata_tmp.obs['GSE'].unique()[0]) & (age_rectify_df['gsm']==adata_tmp.obs['GSM'].unique()[0])]
    if not age_tmp.empty and (age_tmp['age_group'] != 'Unknown').any():  # (age_tmp.shape[0]>0) and (age_tmp['age_group']!='Unknown')
        adata_tmp.obs['age'] = age_tmp['age_final'].to_list()[0] 
        adata_tmp.obs['age_group'] = age_tmp['age_group'].to_list()[0] 
        if (adata_tmp.shape[0] > 50) and ('leiden' in adata_tmp.obs.columns):
            adata_tmp = adata_tmp.raw.to_adata()
            tissue_tmp = tissue_rectify_df[(tissue_rectify_df['gse']==adata_tmp.obs['GSE'].unique()[0]) & (tissue_rectify_df['gsm']==adata_tmp.obs['GSM'].unique()[0])]
            if tissue_tmp.shape[0]>0:
                adata_tmp.obs['Tissue'] = tissue_tmp['tissue_final'].to_list()[0]
            disease_tmp = disease_rectify_df[(disease_rectify_df['gse']==adata_tmp.obs['GSE'].unique()[0]) & (disease_rectify_df['gsm']==adata_tmp.obs['GSM'].unique()[0])]
            if disease_tmp.shape[0]>0:
                adata_tmp.obs['disease_global'] = disease_tmp['Global_type'].to_list()[0]
                adata_tmp.obs['disease_anatomical'] = disease_tmp['Anatomical_type'].to_list()[0]
            treat_tmp = treatment_rectify_df[(treatment_rectify_df['gse']==adata_tmp.obs['GSE'].unique()[0]) & (treatment_rectify_df['gsm']==adata_tmp.obs['GSM'].unique()[0])]
            if treat_tmp.shape[0]>0:
                adata_tmp.obs['Treat'] = treat_tmp['treat_final'].to_list()[0]
            platform_tmp = platform_rectify_df[(platform_rectify_df['gse']==adata_tmp.obs['GSE'].unique()[0]) & (platform_rectify_df['gsm']==adata_tmp.obs['GSM'].unique()[0])]
            if platform_tmp.shape[0]>0:
                adata_tmp.obs['Platform'] = platform_tmp['platform_final'].to_list()[0]  
            race_tmp = race_rectify_df[(race_rectify_df['gse']==adata_tmp.obs['GSE'].unique()[0]) & (race_rectify_df['gsm']==adata_tmp.obs['GSM'].unique()[0])]
            if race_tmp.shape[0]>0:
                adata_tmp.obs['Strain'] = race_tmp['strain'].to_list()[0]    
            sex_tmp = reserve_samples[(reserve_samples['gse']==adata_tmp.obs['GSE'].unique()[0]) & (reserve_samples['gsm']==adata_tmp.obs['GSM'].unique()[0])]  ## sex_rectify_df
            if sex_tmp.shape[0]>0:
                adata_tmp.obs['Sex'] = sex_tmp['sex_predict'].to_list()[0]  
            adata_new = adata_tmp
            adata_var_index_upper = [x.upper() for x in adata_new.var.index]  # adata_tmp.raw.shape # 这样返回的是没取HVG = 2000 的结果
            Endo_union_gene_inter = list(set(adata_var_index_upper).intersection(Endo_marker_union))
            index_in_df = [adata_var_index_upper.index(x) for x in Endo_union_gene_inter]
            Endo_union_geneset = adata_new.var.iloc[index_in_df,].index.to_list()
            if len(Endo_union_geneset) > 0:
                sc.tl.score_genes(adata_new, Endo_union_geneset, score_name='MS_Endo_union', use_raw=False)  ### 没有这个函数  sc.tl.score_geneset
            ms_columns = [col for col in adata_new.obs.columns if 'MS_' in col]
            grouped_leiden_all = adata_new.obs.groupby('leiden')[ms_columns].describe()  ### adata
            grouped_leiden_all.columns = grouped_leiden_all.columns.droplevel(0)
            grouped_leiden_all.index = grouped_leiden_all.index.astype(int)
            grouped_leiden_all.index.name = None
            # grouped_leiden_all = adata.obs.groupby('leiden')[['MS_Fib_general','MS_Fib_spec','MS_Fib_yilv','MS_Fib_union','MS_Endo_general','MS_Endo_spec','MS_Endo_union','MS_MoMac_general','MS_MoMac_spec','MS_MoMac_union']].describe()  ### adata.obs.groupby
            species = dataset_file.split('/')[3]
            thresh_tmp = 0.2
            ms_columns_endo = [item for item in ms_columns if 'Endo' in item]
            cluster_index = grouped_leiden_all.index[grouped_leiden_all['mean']> thresh_tmp].tolist()
            if (len(cluster_index)>=1):
                cluster_index = [str(num) for num in cluster_index]
                adata_sub = adata_new[adata_new.obs['leiden'].isin(cluster_index), :]
                cell_num = adata_sub.shape[0]
                gse = adata_sub.obs['GSE'].unique()[0]
                gsm = adata_sub.obs['GSM'].unique()[0]
                cluster_num = len(cluster_index)
                cell_lineage = 'Endo'
                data={'gse':gse,'gsm':gsm,'lineage':cell_lineage,'cell_num':cell_num, 'clus_num':cluster_num}
                df_tmp = pd.DataFrame([data])
                adata_summary_df = adata_summary_df._append(df_tmp)  
                adata_list_merge.append(adata_sub)  


In [ ]:
adata_summary_df

In [ ]:
adata_summary_df['cell_num'].sum()


In [ ]:
Endo_human_cell_num_blood_sub_new_1[['gse','gsm','cell_num']].drop_duplicates()['cell_num'].sum()

In [ ]:
Endo_human_cell_num_blood_sub_new_1[Endo_human_cell_num_blood_sub_new_1['gse'].isin(['GSE132950','GSE149356'])]


In [ ]:
Fib_human_meta

In [ ]:
Fib_human_meta['cell_num'].sum()

In [ ]:
Momac_human_meta['cell_num'].sum() # 3,789,382


In [ ]:
sc.pl.dotplot(adata_tmp, var_names=list(set(list(adata_tmp.var_names)) & set(Fib_Endo_marker_union)) , groupby='leiden', dendrogram=False)

In [ ]:
############################################################ 尝试用 dynamo 做 trajectory分析, 这个软件很慢, 所以用down sampling的数据来做       ###########################
import dynamo as dyn
import stream
adata = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_GSMbatch_annotation_ds_5k.h5ad')
adata


In [ ]:
preprocessor = dyn.pp.Preprocessor(force_gene_list=selected_genes_to_use)
preprocessor.config_monocle_recipe(adata, n_top_genes=len(selected_genes_to_use))
preprocessor.preprocess_adata(
    adata, recipe='monocle'
)
# preprocessor = dyn.pp.Preprocessor(cell_cycle_score_enable=True)
# preprocessor.preprocess_adata(adata, recipe='monocle')


In [ ]:
dyn.pl.umap(adata, color='Celltype')

In [ ]:
dyn.pl.umap(adata, color='leiden')

In [ ]:
help(preprocessor.preprocess_adata)


In [ ]:
from dynamo.preprocessing import Preprocessor
preprocessor = Preprocessor()
preprocessor.preprocess_adata(adata, recipe="monocle")

# Alternative
# preprocessor.config_monocle_recipe(adata)
# preprocessor.preprocess_adata_monocle(adata)

# default_preprocessor_monocle_adata = adata # save for usage later


In [ ]:
adata_a = dyn.sample_data.zebrafish()
figsize = (10, 10)
celltype_key = 'Cell_type'

preprocessor = Preprocessor()
preprocessor.preprocess_adata(adata_a, recipe="monocle")

# Alternative
# preprocessor.config_monocle_recipe(adata)
# preprocessor.preprocess_adata_monocle(adata)

default_preprocessor_monocle_adata = adata_a # save for usage later

dyn.tl.reduceDimension(adata_a, basis="pca")
dyn.pl.umap(adata_a, color=celltype_key, figsize=(4,4),
           adjust_legend=True)


In [ ]:
adata_a.X.toarray()[0:10,0:10]



In [ ]:
cluster_to_celltype = {
    '0': 'eFibro_RPL31',
    '1': 'Myfibro_MYH11', ### MyoFibroblast 已经是比较terminal的
    '2': 'eFibro_RPL31',
    '3': 'qFibro_LUM',  # LUM
    '4': 'apFibro_JUNB',
    '5': 'qFibro_COL1A1',   # COL1A1
    '6': 'eFibro_CTHRC1',
    '7': 'qFibro_DCN',   # DCN 
    '8': 'Myfibro_FTX',  # FTX, 癌症相关的 LNC RNA, Myo caf, MyCAF
    '9': 'Myfibro_MDK',  # MDK, 与 Myfibro_MYH11 很相似
    '10': 'eFibro_RPL31',  # H2AZ1, HMGB2
    '11': 'apFibro_RPL31', # CD7, MZB1, IGLL1
    '12': 'eFibro_CFD',
    '13': 'iFibro_KRT14',
    '14': 'apFibro_CD74',
    '15': 'iFibro_TNFAIP6',  # TNFAIP6, SOD2
    '16': 'eFibro_PTGDS',  # PTGDS
    '17': 'iFibro_CLU', # CLU, TUBB2B, Immu-caf
    '18': 'qFibro_APOA1', # APOA1, AMH, 有ECM，也有immune regulation
    '19': 'apFibro_TSPAN1', # C20orf85, CAPS, TSPAN1; 顾云燕老师的 crFibro_TSPAN8
    '20': 'apFibro_NKG7', # NKG7, PTPRC, CST7
}
adata.obs['Celltype'] = adata.obs['leiden'].map(cluster_to_celltype)
unmapped_clusters = set(adata.obs['leiden']) - set(cluster_to_celltype.keys())
if unmapped_clusters:
    print(f"未定义的聚类: {unmapped_clusters}")
    adata.obs['Celltype'] = adata.obs['leiden'].map(lambda x: cluster_to_celltype.get(x, 'Unknown'))


In [ ]:
os.chdir('./data/scalex/fibro/human/')
cell = pd.DataFrame({'Cell':adata.obs['Celltype'].to_list()})
cell.shape
cell.to_csv('./data/scalex/fibro/human/stream_result/celltype_ds.tsv',index=False, sep='\t',header=False)


In [ ]:
df1.to_csv('./data/scalex/fibro/human/stream_result/leiden_color.tsv',index=False, sep='\t',header=False)
df2.to_csv('./data/scalex/fibro/human/stream_result/celltype_color.tsv',index=False, sep='\t',header=False)


In [ ]:
import stream as st
adata.var_names_make_unique()
adata.obs_names_make_unique()
st.add_cell_labels(adata,file_name='./data/scalex/fibro/human/stream_result/celltype_ds.tsv')
st.add_cell_colors(adata,file_name='./data/scalex/fibro/human/stream_result/celltype_color.tsv')
st.cal_qc(adata,assay='rna')
st.filter_features(adata,min_n_cells = 5)
st.filter_cells(adata,min_n_features= 100)
st.filter_features(adata,min_n_cells = 5)
###Normalize gene expression based on library size  
#st.normalize(adata,method='lib_size')
###Logarithmize gene expression 
#st.log_transform(adata)
###Remove mitochondrial genes 
# st.remove_mt_genes(adata)


In [ ]:
st.select_variable_genes(adata,loess_frac=0.01)
